# 00 — Setup & Chat Model Basics

**LangChain version this cookbook targets:** `langchain>=1.0` (v1.0 went GA October 22, 2025).

### Why a fresh cookbook matters
Most LangChain tutorials on the internet still use pre-1.0 patterns — `LLMChain`, `AgentExecutor`, `ConversationBufferMemory` — which are **deprecated** and now live in a separate `langchain-classic` package. If a tutorial imports from `langchain.chains.LLMChain`, it's stale. This cookbook only uses current, supported APIs.

### What changed in v1.0 that you should know upfront
1. **Provider packages are split out.** You don't get OpenAI/Anthropic support from `langchain` itself anymore — you install `langchain-openai`, `langchain-anthropic`, etc. separately.
2. **`create_agent`** (in `langchain.agents`) is now the standard way to build an agent — replacing the old `AgentExecutor` + `initialize_agent` pattern.
3. **LangGraph is the runtime underneath.** Even the "simple" LangChain agent API is built on LangGraph, so memory/state/persistence concepts you learn here carry over directly if you go deeper into LangGraph later.
4. **LCEL (LangChain Expression Language)** — the `prompt | model | parser` pipe syntax — is still the standard way to build simple chains. This did *not* change in v1.0.

### Notebook map (run these in order the first time)
| # | Notebook | Concept |
|---|----------|---------|
| 00 | This one | Setup, chat models, invoke/stream |
| 01 | `01_prompts_and_chains.ipynb` | Prompt templates, LCEL chains, output parsers |
| 02 | `02_tools_and_agents.ipynb` | Tool calling, `create_agent` |
| 03 | `03_memory_and_state.ipynb` | Multi-turn memory via checkpointer |
| 04 | `04_structured_output.ipynb` | Pydantic structured output (extraction) |
| 05 | `05_rag_pipeline.ipynb` | Document loading, embeddings, retrieval, RAG chain |


## 1. Install dependencies

Run this once. These are the exact packages/versions this cookbook was validated against as of **July 2026**:

```
langchain==1.3.14
langchain-openai==1.4.1
langchain-anthropic==1.5.2
langgraph==1.2.9
langchain-text-splitters==1.1.2
```

Newer patch versions should be fine — LangChain committed to no breaking changes within the 1.x line until 2.0.


In [1]:
%pip install -q langchain langchain-openai langchain-anthropic langgraph langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


## 2. Set your API key

This cookbook is written provider-agnostic — swap between OpenAI and Anthropic by changing **one line** (`MODEL_ID` below). You only need a key for whichever provider you pick.

Get a key:
- OpenAI: https://platform.openai.com/api-keys
- Anthropic: https://console.anthropic.com/settings/keys

**Never hardcode your key in the notebook.** Use `getpass` so it isn't saved into the `.ipynb` file's output.


In [2]:
import os
from getpass import getpass

# Pick ONE provider by uncommenting it. This single choice propagates through every notebook in the cookbook.

# --- Option A: OpenAI ---
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OPENAI_API_KEY: ")
MODEL_ID = "openai:gpt-4.1-mini"   # cheap + fast, good for learning. Swap for "openai:gpt-4.1" if you want stronger reasoning.

# --- Option B: Anthropic (comment out Option A above, uncomment below) ---
# if not os.environ.get("ANTHROPIC_API_KEY"):
#     os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your ANTHROPIC_API_KEY: ")
# MODEL_ID = "anthropic:claude-sonnet-4-5"

print("Using model:", MODEL_ID)

Enter your OPENAI_API_KEY:  ········


Using model: openai:gpt-4.1-mini


## 3. Your first chat model call

LangChain v1.0 lets you reference a model with a simple `"provider:model_name"` string — it resolves to the right integration package behind the scenes (this needs `langchain[openai]`/`langchain-openai` etc. installed, which we did above).

`init_chat_model` is the current, recommended entry point — it replaces provider-specific imports like `ChatOpenAI(...)` when you want your code to stay provider-agnostic.


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(MODEL_ID, temperature=0.3)

response = model.invoke("In one sentence, what is LangChain for?")
print(response.content)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### What did `.invoke()` return?
It's an `AIMessage` object, not a raw string. This matters once you build multi-turn conversations — you'll pass lists of typed messages (`HumanMessage`, `AIMessage`, `SystemMessage`) back and forth.


In [ ]:
print(type(response))
print(response.response_metadata.get("model_name", "n/a"))
print(response.usage_metadata)  # token counts — useful for cost tracking

## 4. Multi-turn conversation with typed messages

No memory system needed yet — just pass a growing list of messages. This is the foundation everything else (agents, memory, RAG) builds on.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a terse, no-nonsense senior engineer. Answer in <=2 sentences."),
    HumanMessage("What's the difference between LangChain and LangGraph?"),
]

reply = model.invoke(conversation)
print(reply.content)

# Append the AI's reply and continue the conversation
conversation.append(reply)
conversation.append(HumanMessage("Which one should I learn first?"))

reply2 = model.invoke(conversation)
print()
print(reply2.content)

## 5. Streaming

For anything user-facing, you generally want to stream tokens rather than wait for the full response.


In [ ]:
for chunk in model.stream("List 3 good reasons to use type hints in Python. Keep it brief."):
    print(chunk.content, end="", flush=True)

---
### Key takeaways
- Install split provider packages (`langchain-openai`, `langchain-anthropic`), not just `langchain`.
- Use `init_chat_model("provider:model")` to stay provider-agnostic.
- `.invoke()` returns a typed `AIMessage`; `.stream()` yields chunks.
- Conversations are just growing lists of `HumanMessage` / `AIMessage` / `SystemMessage`.

**Next:** `01_prompts_and_chains.ipynb` — reusable prompt templates and LCEL chains.
